<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Lab_1_Lektion_4_Enkel_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Städrobot – Lektion 4: Enkel AI (Regelbaserad agent)

I den här lektionen ska vi låta roboten **tänka och fatta egna beslut**!

Hittills har roboten bara känt av världen (**perception**).  
Nu kombinerar vi perception med **regler** – och roboten börjar agera på egen hand.

**Mål för lektionen:**
- 🧠 Förstå vad en *agent* är
- 📋 Skriva `if`-satser baserade på perception
- 🔄 Förstå agent-loopen: perception → beslut → handling
- 🏃 Se roboten lösa problemet automatiskt

> 📌 **Kräver Lektion 3:** Vi använder `get_perception()` och de tre sensorfunktionerna härifrån.

## ⚙️ Setup – Ladda in allt vi behöver
Kör den här cellen **först**.

In [ ]:
# ============================================================
# SETUP – Kör den här cellen först!
# ============================================================
import random  # Behövs för slumpmässiga val

# Celltyper
EMPTY = 0
WALL  = 1
DIRT  = 2
HOME  = 3

SYMBOLS = {EMPTY: ' . ', WALL: ' # ', DIRT: ' ░ ', HOME: ' H '}

# Exempelvärld (6x6)
world = [
    [WALL,  WALL,  WALL,  WALL,  WALL,  WALL],
    [WALL,  HOME,  EMPTY, DIRT,  EMPTY, WALL],
    [WALL,  EMPTY, WALL,  EMPTY, DIRT,  WALL],
    [WALL,  EMPTY, EMPTY, EMPTY, EMPTY, WALL],
    [WALL,  DIRT,  EMPTY, WALL,  EMPTY, WALL],
    [WALL,  WALL,  WALL,  WALL,  WALL,  WALL],
]

home_x, home_y = 1, 1

# Sensorerna från Lektion 3
def can_see_dirt(robot_x, robot_y, world):
    return world[robot_y][robot_x] == DIRT

def is_at_home(robot_x, robot_y, home_x, home_y):
    return robot_x == home_x and robot_y == home_y

def can_move_to(x, y, world):
    if y < 0 or y >= len(world): return False
    if x < 0 or x >= len(world[0]): return False
    return world[y][x] != WALL

def get_perception(robot_x, robot_y, world, home_x, home_y):
    return {
        'dirt_here':      can_see_dirt(robot_x, robot_y, world),
        'at_home':        is_at_home(robot_x, robot_y, home_x, home_y),
        'can_move_right': can_move_to(robot_x + 1, robot_y, world),
        'can_move_left':  can_move_to(robot_x - 1, robot_y, world),
        'can_move_up':    can_move_to(robot_x, robot_y - 1, world),
        'can_move_down':  can_move_to(robot_x, robot_y + 1, world),
    }

# Rörelsefunktion
def move_robot(robot_x, robot_y, direction, world):
    """Utför en rörelse. Returnerar ny (x, y) och poängändring."""
    nx, ny = robot_x, robot_y
    if direction == 'HÖGER':   nx += 1
    elif direction == 'VÄNSTER': nx -= 1
    elif direction == 'UPP':     ny -= 1
    elif direction == 'NER':     ny += 1
    # Kolla om rörelsen är giltig
    if can_move_to(nx, ny, world):
        return nx, ny, -1   # Rörde sig – kostnad: -1 poäng
    else:
        return robot_x, robot_y, -1  # Krock med vägg – kvar på plats

def suck_dirt(robot_x, robot_y, world):
    """Dammsug om det finns smuts. Returnerar poängändring."""
    if world[robot_y][robot_x] == DIRT:
        world[robot_y][robot_x] = EMPTY  # Ta bort smutsen!
        return 100    # Städade – bonus: +100 poäng
    return -1         # Ingen smuts – kostnad: -1 poäng

def is_all_clean(world):
    """Returnerar True om det inte finns någon smuts kvar."""
    for rad in world:
        for cell in rad:
            if cell == DIRT:
                return False
    return True

def visa_världen(world, robot_x, robot_y):
    print("\n  Kolumn: 0    1    2    3    4    5")
    for y, rad in enumerate(world):
        print(f"Rad {y}: ", end="")
        for x, cell in enumerate(rad):
            if x == robot_x and y == robot_y:
                print(' R ', end=" ")
            else:
                print(SYMBOLS[cell], end=" ")
        print()
    print()

print("Setup klar! Världen är redo.")
visa_världen(world, home_x, home_y)

---
## Del 1: Vad är en agent? 🤖

En **agent** är något som:
1. **Uppfattar** sin omgivning (perception)
2. **Bestämmer** sig för vad den ska göra (beslut)
3. **Utför** en handling
4. **Upprepar** – börjar om från punkt 1

Det kallas för **agent-loopen**:
```
┌─────────────────────────────────────────────┐
│  Perception → Beslut → Handling → Perception │
└─────────────────────────────────────────────┘
```

**Regler** är det som styr beslutet:  
> *"Om jag ser smuts → dammsug"*  
> *"Om jag är hemma och allt är städat → stanna"*

En agent som styrs av regler kallas en **regelbaserad agent**.

---
## Del 2: En enkel regel – "Om smuts → dammsug" 🧹

Vi börjar med den enklaste möjliga agenten: **en regel**.

```
Regel 1: Om det finns smuts → dammsug
Annars:  Gå höger
```

In [ ]:
# Den enklaste möjliga agenten
# ----------------------------
def enkel_agent(perception):
    """Robotens hjärna – en mycket enkel beslutsregel."""
    if perception['dirt_here']:  # Om det finns smuts...
        return 'DAMMSUG'         # ...dammsug!
    else:
        return 'HÖGER'           # Annars: gå höger

# Testa agenten med olika perceptioner
print("=== Testa enkel_agent ===")

# Scenario 1: Roboten ser smuts
p1 = get_perception(3, 1, world, home_x, home_y)
beslut1 = enkel_agent(p1)
print(f"Scenario 1 – position (3,1), smuts={p1['dirt_here']}")
print(f"  Beslut: {beslut1}")

# Scenario 2: Roboten ser ingen smuts
p2 = get_perception(2, 1, world, home_x, home_y)
beslut2 = enkel_agent(p2)
print(f"\nScenario 2 – position (2,1), smuts={p2['dirt_here']}")
print(f"  Beslut: {beslut2}")

### 🤔 Diskussion

- Vad gör roboten när det **inte** finns smuts?
- Vad händer när roboten nått **höger vägg**?
- Kan den här agenten **städa hela rummet**? Varför/varför inte?

---
## Del 3: Fler regler – "smart_agent" 🧠

Den enkla agenten fastnar vid höger vägg. Låt oss lägga till fler regler:

```
Regel 1: Om hemma OCH allt städat → stanna
Regel 2: Om smuts → dammsug
Regel 3: Gå slumpmässigt i en fri riktning
```

Reglerna kollas **i ordning** – den **första** som stämmer vinner.

In [ ]:
# En smartare agent med flera regler
# ------------------------------------
def smart_agent(perception, all_clean):
    """En agent med fyra regler – körs i ordning."""

    # Regel 1: Uppdrag klart – stanna!
    if perception['at_home'] and all_clean:
        return 'STANNA'

    # Regel 2: Ser smuts – dammsug!
    if perception['dirt_here']:
        return 'DAMMSUG'

    # Regel 3: Gå slumpmässigt i en fri riktning
    möjliga_rörelser = []
    if perception['can_move_right']:  möjliga_rörelser.append('HÖGER')
    if perception['can_move_left']:   möjliga_rörelser.append('VÄNSTER')
    if perception['can_move_up']:     möjliga_rörelser.append('UPP')
    if perception['can_move_down']:   möjliga_rörelser.append('NER')

    if möjliga_rörelser:
        return random.choice(möjliga_rörelser)  # Välj en slumpmässig riktning
    else:
        return 'STANNA'  # Ingen riktning möjlig

# Testa smart_agent
print("=== Testa smart_agent ===")

test_positioner = [(3, 1), (1, 1), (2, 3)]
for (rx, ry) in test_positioner:
    p = get_perception(rx, ry, world, home_x, home_y)
    allt_städat = is_all_clean(world)
    beslut = smart_agent(p, allt_städat)
    print(f"Position ({rx},{ry}): smuts={p['dirt_here']}, hemma={p['at_home']} → {beslut}")

---
## Del 4: Agent-loopen – Automatisk körning 🔄

Nu sätter vi ihop allt till en **loop** som kör agenten automatiskt:

```
Steg 1 → Steg 2 → Steg 3 → ... → Stanna
```

Loopen upprepar sig så länge uppdraget inte är klart (eller tills vi når max antal steg).

In [ ]:
# Agent-loop – kör agenten automatiskt
# --------------------------------------
def kör_agent(agent_funktion, max_steg=50, visa_varje_steg=True):
    """
    Kör en agent i en loop tills uppdraget är klart eller max_steg nås.
    Returnerar slutpoängen.
    """
    # Skapa en kopia av världen (så vi inte förstör originalet)
    import copy
    min_värld = copy.deepcopy(world)

    # Starttillstånd
    rx, ry  = home_x, home_y  # Robot börjar hemma
    poäng   = -1000            # Startpoäng – roboten börjar med -1000 i 'skuld' som den städar bort
    steg    = 0

    print(f"🚀 Startar agent: {agent_funktion.__name__}")
    print(f"   Startposition: ({rx}, {ry}), Startpoäng: {poäng}")

    while steg < max_steg:
        steg += 1

        # --- Steg 1: Perception ---
        p = get_perception(rx, ry, min_värld, home_x, home_y)
        allt_städat = is_all_clean(min_värld)

        # --- Steg 2: Beslut ---
        åtgärd = agent_funktion(p, allt_städat)

        # --- Steg 3: Handling ---
        if åtgärd == 'STANNA':
            print(f"\n✅ Steg {steg}: STANNA – uppdrag klart!")
            break
        elif åtgärd == 'DAMMSUG':
            poäng_ändring = suck_dirt(rx, ry, min_värld)
            poäng += poäng_ändring
        else:  # Rörelsekommando
            rx, ry, poäng_ändring = move_robot(rx, ry, åtgärd, min_värld)
            poäng += poäng_ändring

        # --- Steg 4: Visualisering ---
        if visa_varje_steg:
            print(f"Steg {steg:3}: {åtgärd:10} → ({rx},{ry}), poäng: {poäng:6}")

    print(f"\n📊 Resultat: {steg} steg, slutpoäng: {poäng}")
    return poäng

# Kör den smarta agenten!
random.seed(42)  # Fixerar slumpen för reproducerbarhet
kör_agent(smart_agent, max_steg=30, visa_varje_steg=True)

### 🔬 Experiment 4.1 – Kör med olika frön

Prova att ändra `random.seed(...)` till olika värden (t.ex. 1, 7, 99).  
Hur många steg tar agenten varje gång? Varför varierar det?

In [ ]:
# Experiment 4.1 – Jämför körningar med olika slump
for frö in [1, 7, 42, 99]:
    import copy
    random.seed(frö)
    print(f"\n--- Frö: {frö} ---")
    kör_agent(smart_agent, max_steg=60, visa_varje_steg=False)

---
## Del 5: Experimentera med regler ✏️

### Övning 4.1 – Ändra regelordning

Vad händer om vi kontrollerar **hemma** FÖRE **smuts**?
Fyll i agenten nedan med omvänd regelordning och testa.

In [ ]:
# Övning 4.1 – Agenten med omvänd regelordning
def agent_omvänd(perception, all_clean):
    """Regelordning: hemma → smuts → slump."""

    # Regel 1: Om hemma → stanna (oavsett smuts!)
    if perception['at_home']:
        return ___  # ← Fyll i: vad ska agenten göra hemma?

    # Regel 2: Om smuts → dammsug
    if perception['dirt_here']:
        return ___  # ← Fyll i

    # Regel 3: Slumpmässig riktning
    möjliga = [r for r in ['HÖGER','VÄNSTER','UPP','NER']
               if perception[f'can_move_{r.lower()}'] or
                  perception.get(f'can_move_{["right","left","up","down"][["HÖGER","VÄNSTER","UPP","NER"].index(r)]}')]
    möjliga = []
    if perception['can_move_right']: möjliga.append('HÖGER')
    if perception['can_move_left']:  möjliga.append('VÄNSTER')
    if perception['can_move_up']:    möjliga.append('UPP')
    if perception['can_move_down']:  möjliga.append('NER')
    return random.choice(möjliga) if möjliga else 'STANNA'

# Kör och jämför
random.seed(42)
print("Agent med omvänd regelordning:")
kör_agent(agent_omvänd, max_steg=20, visa_varje_steg=True)

### Övning 4.2 – Lägg till en ny regel

Lägg till regeln: **"Gå aldrig tillbaka till förra positionen"**  
Spara förra positionen i en variabel och undvik den riktningen.

In [ ]:
# Övning 4.2 – Agent som undviker att gå tillbaka
def agent_inget_tillbaka(perception, all_clean, förra_riktning=None):
    """Undviker att göra samma rörelse som senast."""

    # Regel 1: Uppdrag klart
    if perception['at_home'] and all_clean:
        return 'STANNA'

    # Regel 2: Dammsug
    if perception['dirt_here']:
        return 'DAMMSUG'

    # Regel 3: Välj riktning (undvik förra)
    MOTSATT = {'HÖGER': 'VÄNSTER', 'VÄNSTER': 'HÖGER',
               'UPP': 'NER', 'NER': 'UPP'}

    möjliga = []
    if perception['can_move_right']: möjliga.append('HÖGER')
    if perception['can_move_left']:  möjliga.append('VÄNSTER')
    if perception['can_move_up']:    möjliga.append('UPP')
    if perception['can_move_down']:  möjliga.append('NER')

    # Ta bort den motsatta riktningen (undvik att gå tillbaka)
    if förra_riktning and MOTSATT[förra_riktning] in möjliga and len(möjliga) > 1:
        möjliga.remove(MOTSATT[förra_riktning])

    return random.choice(möjliga) if möjliga else 'STANNA'

# Testa
print("Test: agent_inget_tillbaka (ingen förra riktning)")
p = get_perception(2, 3, world, home_x, home_y)
print(f"Beslut: {agent_inget_tillbaka(p, False, None)}")
print(f"Beslut (förra=HÖGER): {agent_inget_tillbaka(p, False, 'HÖGER')}")

### Övning 4.3 – Jämför agenter

Kör tre versioner av agenten och jämför resultaten.

In [ ]:
# Övning 4.3 – Jämför tre agenter
# Enkel agent (bara två regler)
def enkel_agent_v2(perception, all_clean):
    """Enkel agent: dammsug om smuts, annars slumpmässig riktning."""
    if perception['dirt_here']:
        return 'DAMMSUG'
    möjliga = []
    if perception['can_move_right']: möjliga.append('HÖGER')
    if perception['can_move_left']:  möjliga.append('VÄNSTER')
    if perception['can_move_up']:    möjliga.append('UPP')
    if perception['can_move_down']:  möjliga.append('NER')
    return random.choice(möjliga) if möjliga else 'STANNA'

# Kör alla tre och jämför (använd samma frö)
print("Jämförelse av tre agenter (frö=42):")
print("="*40)

for agent_fn in [enkel_agent_v2, smart_agent]:
    random.seed(42)
    print(f"\nAgent: {agent_fn.__name__}")
    poäng = kör_agent(agent_fn, max_steg=60, visa_varje_steg=False)
    print(f"Slutpoäng: {poäng}")

### Övning 4.4 – Skriv din egen agent

Kan du skriva en agent som är **bättre** än `smart_agent`?  

**Tips:**
- Gå alltid mot hemmet om allt är städat
- Undvik att gå tillbaka dit du precis kom ifrån
- Prioritera rörelser mot oexplorade delar

In [ ]:
# Övning 4.4 – Skriv din egen agent!
def min_agent(perception, all_clean):
    """Min egen agent – fyll i reglerna!"""

    # Regel 1: ?
    # ...

    # Regel 2: ?
    # ...

    # Fallback: stanna
    return 'STANNA'

# Testa din agent!
random.seed(42)
kör_agent(min_agent, max_steg=60, visa_varje_steg=True)

---
## Del 6: Begränsningar med regelbaserade agenter ⚠️

Den regelbaserade agenten är enkel och förståelig – men den har **svagheter**:

| Problem | Förklaring |
|---------|------------|
| Kan fastna i loopar | Roboten pendlar fram och tillbaka |
| Ingen planering | Den vet inte var smutsen finns |
| Slumpmässig | Prestandan varierar varje körning |
| Ingen karta | Den vet inte var den har varit |

🔜 **Nästa steg (Lektion 5):** Vi introducerar **sökalgoritmerna** DFT och BFS  
som låter roboten planera en **effektiv väg** istället för att gå slumpmässigt!

In [ ]:
# Demo: Hur lång tid tar det för smart_agent att städa?
# Kör agenten 10 gånger med olika slumpfrön
resultat = []
for frö in range(10):
    import copy
    random.seed(frö)

    # Kör agent (utan utskrift)
    min_värld = copy.deepcopy(world)
    rx, ry = home_x, home_y
    poäng = -1000
    steg = 0
    klar = False

    while steg < 200 and not klar:
        steg += 1
        p = get_perception(rx, ry, min_värld, home_x, home_y)
        allt = is_all_clean(min_värld)
        åtgärd = smart_agent(p, allt)
        if åtgärd == 'STANNA':
            klar = True
        elif åtgärd == 'DAMMSUG':
            poäng += suck_dirt(rx, ry, min_värld)
        else:
            rx, ry, dpoäng = move_robot(rx, ry, åtgärd, min_värld)
            poäng += dpoäng

    status = '✅' if klar else '⏰ timeout'
    resultat.append(steg)
    print(f"Frö {frö:2}: {steg:4} steg, poäng: {poäng:6} {status}")

print(f"\nMedel: {sum(resultat)/len(resultat):.1f} steg")
print(f"Min:   {min(resultat)} steg, Max: {max(resultat)} steg")

---
## Del 7: Quiz – Testa dina kunskaper! 🎓

In [ ]:
# QUIZ – Fyll i svaren (ersätt ??? med ditt svar)

# Fråga 1: Vad är de tre stegen i agent-loopen?
svar_1 = ???  # 'perception-beslut-handling', 'input-process-output' etc.

# Fråga 2: Vad returnerar smart_agent() om roboten är hemma OCH allt är städat?
svar_2 = ???  # Strängen agenten returnerar

# Fråga 3: Kan smart_agent() planera en väg till smutsen?
svar_3 = ???  # True eller False

# Fråga 4: Vad händer om inga rörelser är möjliga i smart_agent()?
svar_4 = ???  # Vad returneras?

# Kontrollera
rätt = 0
if isinstance(svar_1, str) and len(svar_1) > 5:  rätt += 1;  print("Q1 ✅")
else:                                             print("Q1 ❌ – Tips: tre ord med bindestreck")

if svar_2 == 'STANNA':  rätt += 1;  print("Q2 ✅")
else:                   print("Q2 ❌ – Titta på Regel 1 i smart_agent")

if svar_3 == False:  rätt += 1;  print("Q3 ✅")
else:                print("Q3 ❌ – Agenten väljer slumpmässigt, ingen planering")

if svar_4 == 'STANNA':  rätt += 1;  print("Q4 ✅")
else:                   print("Q4 ❌ – Kolla 'else'-grenen i smart_agent")

print(f"\nResultat: {rätt}/4 rätt! {'🎉 Utmärkt!' if rätt == 4 else '📚 Fortsätt öva!'}")

---
## 🎯 Sammanfattning – Lektion 4

Du har lärt dig:

| Begrepp | Förklaring |
|---------|------------|
| **Agent** | Något som uppfattar och agerar i sin omgivning |
| **Regelbaserad agent** | Fattar beslut baserat på `if`-regler |
| **Agent-loop** | Perception → Beslut → Handling → Upprepa |
| **`smart_agent()`** | En agent med prioriterade regler |

### Från manuell styrning till AI:
```
Lektion 2: Du styrde roboten manuellt
Lektion 3: Roboten kände av sin omgivning
Lektion 4: Roboten fattar egna beslut ← Du är här!
Lektion 5: Roboten planerar en effektiv väg (DFT/BFS)
```

🔜 **Nästa lektion (Lektion 5):** Sökalgoritmerna – roboten planerar en **optimal väg**!